In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

# Set data path
data_path = Path('data/raw')

# List all CSV files
csv_files = list(data_path.glob('*.csv'))
print(f"Found {len(csv_files)} CSV files:\n")

for file in sorted(csv_files):
    print(f"  • {file.name}")

Found 9 CSV files:

  • olist_customers_dataset.csv
  • olist_geolocation_dataset.csv
  • olist_order_items_dataset.csv
  • olist_order_payments_dataset.csv
  • olist_order_reviews_dataset.csv
  • olist_orders_dataset.csv
  • olist_products_dataset.csv
  • olist_sellers_dataset.csv
  • product_category_name_translation.csv


In [2]:
# Load the orders dataset (main table)
orders = pd.read_csv('data/raw/olist_orders_dataset.csv')

print("Orders Dataset Overview:")
print(f"Shape: {orders.shape}")
print(f"\nColumns: {list(orders.columns)}")
print(f"\nFirst few rows:")
print(orders.head())
print(f"\nData types:")
print(orders.dtypes)
print(f"\nMissing values:")
print(orders.isnull().sum())

Orders Dataset Overview:
Shape: (99441, 8)

Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

First few rows:
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   

  order_status order_purchase_timestamp    order_approved_at  \
0    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15   
1    delivered      2018-07-24 20:41:37  2018-07-26 03:24:27   
2    delivered      2018-08-08 08:38:49  2018-08-08 08:55:23   
3    delivered      2017-11-18 19:28:06  201

In [3]:
# Load all datasets
customers = pd.read_csv('data/raw/olist_customers_dataset.csv')
order_items = pd.read_csv('data/raw/olist_order_items_dataset.csv')
products = pd.read_csv('data/raw/olist_products_dataset.csv')
reviews = pd.read_csv('data/raw/olist_order_reviews_dataset.csv')
sellers = pd.read_csv('data/raw/olist_sellers_dataset.csv')
payments = pd.read_csv('data/raw/olist_order_payments_dataset.csv')
category_translation = pd.read_csv('data/raw/product_category_name_translation.csv')
geolocation = pd.read_csv('data/raw/olist_geolocation_dataset.csv')

# Create a dictionary to store all dataframes
datasets = {
    'orders': orders,
    'customers': customers,
    'order_items': order_items,
    'products': products,
    'reviews': reviews,
    'sellers': sellers,
    'payments': payments,
    'category_translation': category_translation,
    'geolocation': geolocation
}

# Print summary of all datasets
print("DATASET SUMMARY")
print("=" * 60)
for name, df in datasets.items():
    print(f"\n{name.upper()}")
    print(f"  Shape: {df.shape[0]} rows × {df.shape[1]} columns")
    print(f"  Memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

DATASET SUMMARY

ORDERS
  Shape: 99441 rows × 8 columns
  Memory: 58.97 MB

CUSTOMERS
  Shape: 99441 rows × 5 columns
  Memory: 29.62 MB

ORDER_ITEMS
  Shape: 112650 rows × 7 columns
  Memory: 39.43 MB

PRODUCTS
  Shape: 32951 rows × 9 columns
  Memory: 6.79 MB

REVIEWS
  Shape: 99224 rows × 7 columns
  Memory: 42.75 MB

SELLERS
  Shape: 3095 rows × 4 columns
  Memory: 0.66 MB

PAYMENTS
  Shape: 103886 rows × 5 columns
  Memory: 17.81 MB

CATEGORY_TRANSLATION
  Shape: 71 rows × 2 columns
  Memory: 0.01 MB

GEOLOCATION
  Shape: 1000163 rows × 5 columns
  Memory: 145.20 MB


In [4]:
# Data Quality Overview
print("DATA QUALITY ASSESSMENT")
print("=" * 80)

for name, df in datasets.items():
    print(f"\n{name.upper()}")
    print(f"  Rows: {len(df)}")
    print(f"  Columns: {len(df.columns)}")
    
    # Count missing values
    missing = df.isnull().sum().sum()
    missing_pct = (missing / (len(df) * len(df.columns))) * 100
    print(f"  Missing values: {missing} ({missing_pct:.2f}%)")
    
    # Duplicate rows
    duplicates = df.duplicated().sum()
    print(f"  Duplicate rows: {duplicates}")

DATA QUALITY ASSESSMENT

ORDERS
  Rows: 99441
  Columns: 8
  Missing values: 4908 (0.62%)
  Duplicate rows: 0

CUSTOMERS
  Rows: 99441
  Columns: 5
  Missing values: 0 (0.00%)
  Duplicate rows: 0

ORDER_ITEMS
  Rows: 112650
  Columns: 7
  Missing values: 0 (0.00%)
  Duplicate rows: 0

PRODUCTS
  Rows: 32951
  Columns: 9
  Missing values: 2448 (0.83%)
  Duplicate rows: 0

REVIEWS
  Rows: 99224
  Columns: 7
  Missing values: 145903 (21.01%)
  Duplicate rows: 0

SELLERS
  Rows: 3095
  Columns: 4
  Missing values: 0 (0.00%)
  Duplicate rows: 0

PAYMENTS
  Rows: 103886
  Columns: 5
  Missing values: 0 (0.00%)
  Duplicate rows: 0

CATEGORY_TRANSLATION
  Rows: 71
  Columns: 2
  Missing values: 0 (0.00%)
  Duplicate rows: 0

GEOLOCATION
  Rows: 1000163
  Columns: 5
  Missing values: 0 (0.00%)
  Duplicate rows: 261831


In [5]:
# Convert date columns to datetime
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])
reviews['review_creation_date'] = pd.to_datetime(reviews['review_creation_date'])

# Date range analysis
print("DATE RANGES IN DATA")
print("=" * 60)
print(f"\nOrders:")
print(f"  From: {orders['order_purchase_timestamp'].min()}")
print(f"  To:   {orders['order_purchase_timestamp'].max()}")
print(f"  Days: {(orders['order_purchase_timestamp'].max() - orders['order_purchase_timestamp'].min()).days}")

print(f"\nReviews:")
print(f"  From: {reviews['review_creation_date'].min()}")
print(f"  To:   {reviews['review_creation_date'].max()}")

DATE RANGES IN DATA

Orders:
  From: 2016-09-04 21:15:19
  To:   2018-10-17 17:30:18
  Days: 772

Reviews:
  From: 2016-10-02 00:00:00
  To:   2018-08-31 00:00:00


In [6]:
# Key Statistics
print("KEY STATISTICS")
print("=" * 80)

print(f"\nTOTAL ORDERS: {len(orders):,}")
print(f"TOTAL REVENUE: ${order_items['price'].sum():,.2f}")
print(f"AVERAGE ORDER VALUE: ${order_items.groupby('order_id')['price'].sum().mean():,.2f}")

print(f"\nUNIQUE CUSTOMERS: {customers['customer_unique_id'].nunique():,}")
print(f"UNIQUE PRODUCTS: {products['product_id'].nunique():,}")
print(f"UNIQUE SELLERS: {sellers['seller_id'].nunique():,}")

print(f"\nAVERAGE RATING: {reviews['review_score'].mean():.2f} / 5.0")
print(f"TOTAL REVIEWS: {len(reviews):,}")

print(f"\nORDERS BY STATUS:")
print(orders['order_status'].value_counts())

KEY STATISTICS

TOTAL ORDERS: 99,441
TOTAL REVENUE: $13,591,643.70
AVERAGE ORDER VALUE: $137.75

UNIQUE CUSTOMERS: 96,096
UNIQUE PRODUCTS: 32,951
UNIQUE SELLERS: 3,095

AVERAGE RATING: 4.09 / 5.0
TOTAL REVIEWS: 99,224

ORDERS BY STATUS:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [8]:
# List all CSV files in the data folder
csv_files = list(data_path.glob('*.csv'))

In [9]:
orders = pd.read_csv('data/raw/olist_orders_dataset.csv')
print(orders.shape)  # How many rows and columns?
print(orders.head()) # Show first 5 rows
print(orders.dtypes) # What type is each column?
print(orders.isnull().sum()) # How many missing values?

(99441, 8)
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   

  order_status order_purchase_timestamp    order_approved_at  \
0    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15   
1    delivered      2018-07-24 20:41:37  2018-07-26 03:24:27   
2    delivered      2018-08-08 08:38:49  2018-08-08 08:55:23   
3    delivered      2017-11-18 19:28:06  2017-11-18 19:45:59   
4    delivered      2018-02-13 21:18:39  2018-02-13 22:20:29   

  order_delivered_carrier_date order_delivered_customer_date  \
0          2017-10-04 19:55:00           2017-10-10 21:25:13   
1          2018-07-26 14:31:00       

In [10]:
# For each dataset:
print(f"Rows: {len(df)}")  # How much data?
print(f"Columns: {len(df.columns)}")  # How many fields?
print(f"Missing values: {df.isnull().sum().sum()}")  # Gaps?
print(f"Duplicates: {df.duplicated().sum()}")  # Repeats?

Rows: 1000163
Columns: 5
Missing values: 0
Duplicates: 261831


In [11]:
# Convert text dates to datetime format so we can analyze them
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])

# Extract month, year, day of week
orders['order_month'] = orders['order_purchase_timestamp'].dt.to_period('M')
orders['day_of_week'] = orders['order_purchase_timestamp'].dt.day_name()

# Count orders by month/year/day
print(orders['order_month'].value_counts())  # How many orders each month?
print(orders['day_of_week'].value_counts())  # Which day is busiest?

order_month
2017-11    7544
2018-01    7269
2018-03    7211
2018-04    6939
2018-05    6873
2018-02    6728
2018-08    6512
2018-07    6292
2018-06    6167
2017-12    5673
2017-10    4631
2017-08    4331
2017-09    4285
2017-07    4026
2017-05    3700
2017-06    3245
2017-03    2682
2017-04    2404
2017-02    1780
2017-01     800
2016-10     324
2018-09      16
2016-09       4
2018-10       4
2016-12       1
Freq: M, Name: count, dtype: int64
day_of_week
Monday       16196
Tuesday      15963
Wednesday    15552
Thursday     14761
Friday       14122
Sunday       11960
Saturday     10887
Name: count, dtype: int64


In [12]:
# Total revenue = sum of all prices
total_revenue = order_items['price'].sum()  # $13.6M

# Total shipping = sum of all freight
total_shipping = order_items['freight_value'].sum()  # Costs

# Average order value
aov = order_items.groupby('order_id')['price'].sum().mean()  # $137.75

# Revenue by category
revenue_by_category = order_items.merge(products, on='product_id').groupby('product_category_name')['price'].sum()

In [13]:
# Group orders by state
top_states = customers.merge(orders, on='customer_id').groupby('customer_state').size()
# Result: SP=40K orders, RJ=15K orders, etc.

# Group orders by city
top_cities = customers.merge(orders, on='customer_id').groupby('customer_city').size()
# Result: São Paulo=20K, Rio de Janeiro=8K, etc.

# Count total states and cities
unique_states = customers['customer_state'].nunique()  # 27 states
unique_cities = customers['customer_city'].nunique()  # 4,119 cities

In [14]:
# Average rating across all orders
avg_rating = reviews['review_score'].mean()  # 4.09 / 5.0

# Distribution of ratings
print(reviews['review_score'].value_counts().sort_index())
# Result: 5-star=60K, 4-star=15K, 3-star=5K, 2-star=2K, 1-star=1K

# Rating by product category
rating_by_category = reviews.merge(order_items).merge(products).groupby('product_category_name')['review_score'].mean()
# Result: Electronics=4.5⭐, Furniture=3.8⭐, etc.

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64


In [15]:
# Count by payment type
print(payments['payment_type'].value_counts())
# Result: Credit Card=73%, Debit Card=15%, Boleto=12%

# Average order value by payment type
payment_aov = payments.merge(order_items).groupby('payment_type')['payment_value'].mean()
# Result: Credit Card=$150, Debit Card=$120, Boleto=$100

# Installment analysis
cc_payments = payments[payments['payment_type'] == 'credit_card']
avg_installments = cc_payments['payment_installments'].mean()  # 7.5 payments

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64


In [16]:
# Top 20 products by orders
top_products = order_items.merge(products).groupby('product_id').size().sort_values(ascending=False).head(20)
# Shows which products are bestsellers

# Top 15 categories by orders
top_categories = order_items.merge(products).groupby('product_category_name').size().sort_values(ascending=False).head(15)
# Electronics = 15K orders, Furniture = 12K orders, etc.

# Product characteristics
avg_weight = products['product_weight_g'].mean()  # 800g average
avg_photos = products['product_photos_qty'].mean()  # 4 photos average

In [17]:
# For each dataset, count missing values
for dataset_name, df in datasets.items():
    missing_count = df.isnull().sum()  # How many missing per column?
    missing_pct = (missing_count / len(df) * 100)  # What percentage?
    
    if missing_count.sum() > 0:
        print(f"{dataset_name}: {missing_count} missing values")

orders: order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64 missing values
products: product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64 missing values
reviews: review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64 missing values
